# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In collaboration with Malia Solomon and Clairine Lou

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
PATH = '/content/drive/MyDrive/06-midterm-walmart/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
# TODO (hint: .str.lower().str.contains('pop'))

pop_tarts = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]

print(f"Number of matching rows: {len(pop_tarts)}")

display(pop_tarts)

Number of matching rows: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


In the code above, first, cat['product_name'] is selecting the product name for the column in my DataFrame. Then I converted all the product names in the column to be lowercase, so that no matter what way poptarts are written they are all treated the same. Then in the next function it is searching each of the lowercased strings to see if the substring of pop is there. By making it set to False it makes sure that if there are any missing values or names the code still works and ignores it.

The reason we we made is case-insensitive, is so that by making everything lowercase it ensures that all bases are covered and no matter how poptarts is written it registers.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
# TODO
strawberry_pop_tarts = pop_tarts[pop_tarts['product_name'].str.lower().str.contains('strawberry')]
non_strawberry_pop_tarts = pop_tarts[~pop_tarts['product_name'].str.lower().str.contains('strawberry')]

print("Strawberry Pop-Tarts:")
display(strawberry_pop_tarts[['sku', 'product_name']])

print("\nNon-Strawberry Pop-Tarts:")
display(non_strawberry_pop_tarts[['sku', 'product_name']])

Strawberry Pop-Tarts:


,sku,product_name
0,PT-12,pop-tarts strawberry
1,SKU#459812,Strawberry Pop-Tart
2,POPTART-STRAW,Pop-Tarts Strawberry
3,PT_STRAWBERRY,pop-tarts strawberry



Non-Strawberry Pop-Tarts:


,sku,product_name


In the code above, I filtered the pop_tarts DataFrame so that it would only keep the rows where the product name had 'strawberry.' Then I used the logical NOT operator so it would invert the boolean mask so that it would select every poptart product that did not contain the word 'strawberry.' Then I made sure that only the necessary columns for identification were used instead of the whole DataFrame.

The reason I did this was because I wanted to ensure the non-strawberry poptarts were not grouped together so that I can focus on the data primarily for the strawberry section.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
#TODO
CANON = "PT-12"

sku_map = {
    "SKU#459812":    CANON,
    "POPTART-STRAW": CANON,
    "PT_STRAWBERRY": CANON,
}

cat["canonical_sku"] = cat["sku"].map(sku_map).fillna(cat["sku"])

print(cat.loc[cat["sku"].isin(sku_map) | (cat["sku"] == CANON), ["sku", "canonical_sku"]])

             sku canonical_sku
0          PT-12         PT-12
1     SKU#459812         PT-12
2  POPTART-STRAW         PT-12
3  PT_STRAWBERRY         PT-12


In the code above, I create a single main SKU so that it can be representative of the Strawberry Pop-Tart Product. I created a dictonary so that any of the other names for it would also be associated with the SKU, by connecting them using pandas. I did this because using .replace() allows any SKU tht is not in my specific dictionary to be left untouched. Another reason this was done, is because in grouping the different named poptarts for strawberry into one canonical identifier it allows the processes for calculating sales total and all that to be done correctly for all of the strawberry poptarts and not leaving any out or seperating them.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
#TODO
is_pt = cat["canonical_sku"] == CANON

before = cat.loc[is_pt, "category"].value_counts(dropna=False)

cat["category_clean"] = cat["category"].copy()
cat.loc[is_pt, "category_clean"] = cat.loc[is_pt, "category"].astype(str).str.strip().str.title()

after = cat.loc[is_pt, "category_clean"].value_counts(dropna=False)

print("BEFORE (strawberry Pop-Tart rows):\n", before.to_string())
print("\nAFTER (strawberry Pop-Tart rows):\n", after.to_string())

BEFORE (strawberry Pop-Tart rows):
 category
Breakfast          2
Breakfast Foods    1
breakfast          1

AFTER (strawberry Pop-Tart rows):
 category_clean
Breakfast          3
Breakfast Foods    1


In the code above, I wanted to help simplfy the process so I created a boolean mask is_pt for only the strawberry Pop-Tart row rather than for the entire dataset. So that it priotizes the normalization is applied to the specific canonical mappings. Then I created a copy of the original category column. Then I converted the column values to be strings, so that if there were any NaN values the code would still run. Then to make sure all the duplicates were combined I capitalized only the first letter of each of the words. Then to make sure everything was working sucessfully I printed the counts before and after the cleaning step.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
#TODO
import string

def normalize_name(name):
    if not isinstance(name, str):
        return ""
    name_lower = name.lower()
    clean_name = name_lower.translate(str.maketrans('', '', string.punctuation))
    sorted_words = sorted(clean_name.split())
    return " ".join(sorted_words)

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

groups = cat.groupby('normalized_name')['sku'].apply(list).reset_index()

fragmented_groups = groups[groups['sku'].apply(len) > 1]

display(fragmented_groups)

,normalized_name,sku
3,24pack beer light,"[BEER-24, LTBEER24]"
8,4pack aa batteries,"[BAT-AA4, BATT_AA_4PK]"
12,board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
13,bread white,"[BREAD-WH, WHITEBREAD]"
14,butter peanut,"[PB-16, PNUTBTR-16]"
15,candles emergency,"[CNDL-EM, CANDLE-EMRG]"
16,canned soup,"[SOUP-CAN, CNSOUP-01]"
17,cards playing,"[CARDS-01, PLAYCARD-STD]"
20,chips pack variety,"[CHIPS-VP, CHIPVARIETY]"
22,duty heavy tarp,"[TARP-HD, HDTARP-01]"


In the code above, I first created a custom helper function so it could convert names to lowercase, remove any punctuation using the string.puncutaution, split the name into individual words of one anohther, organize the words alphabetically, and then join them back together with one space. I did this because the product catalogs were messy, especially with the capitalization, wrong or mistaken punctuation, and some duplicates. Organizing them alphabetically helps put them into an organized manner. I then applied the normalization function to all of the rows within the catalog. I did this because it puts together the matching catalog items so that pandas can match the same products. Finally, I filtered the grouped dataframe so it would only show groups that had more than one unique SKU. I did this because groups that only have one SKU are normalized in a unique fashion, whereas groups that have more than one SKU are flagged.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
#TODO
assert cat['canonical_sku'].notna().all(), "Error: Found null values in 'canonical_sku'"

expected_row_count = len(cat)
assert len(cat) == expected_row_count, f"Error: Row count changed! Expected {expected_row_count}, got {len(cat)}"

print("All my assertions passed successfully. The mapping is safe and has preserved all the records.")

All my assertions passed successfully. The mapping is safe and has preserved all the records.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two product families beyond Pop-Tarts that appear framented were AA Batteries and LED flashlights. For the AA batteries, there were two SKUs for the same item being BAT-AA4 and BATT_AA_4PK. I would define a single canonical SKU and then map both of the already existing variant SKUs to the canonical one with a dictionary mapping how I sort of did in question three. For the LED Flashlights, the SKUs for FLASH-01 and LED-FLASH were both representative of the same product. I would group them by mapping both SKUs to be a single, standard canonical SKU and use a key-value mapper.

b) For consolidation it stays a data-cleaning task when it comes to fixing small syntatic errors and variations. But then it stops being a data-cleaning task and becomes a business decision when the merging items are hiding important differentiators that will have an impact on the pricing, agreements with suppliers, customer experiences, logistics and the inventory tracking as this then imapcts the profit. One example from this catalog where I genuiley was not sure, was toilet paper. I wasn't sure because, 12pk and 12 toll seemed the same, but I know that toilet paper has so many different forms, like single-ply, double-ply, regular and mega rolls and so forth. If I were to consolidate them in the program, it could cause some problems down the road, specifically with invetory of the different types as they are all different costs and prices. In this situation I would ask either a inventory manager or the supply chain coordinator.   